# DermaVision AI — Module 4: Sequential Deep Learning & Longitudinal Skin Monitoring

**Multi-Horizon Trajectory Forecasting (LSTM / GRU) & Clinical Feedback NLP (BiLSTM)**

- **Author:** Mouli Mukherjee (Module 4 Lead)
- **Project:** DermaVision AI — Intelligent Skin Pigmentation Analysis & Disease Detection
- **Curriculum Alignment:** RNN, LSTM, GRU, BiLSTM

---

### Overview
Module 4 introduces **temporal intelligence** to DermaVision AI:
1. **Longitudinal Skin Tracking (LSTM & GRU):** Predicts skin health and pigmentation percentage trajectories at **1, 3, and 6 months** ahead.
2. **Trend Risk Categorization:** Classifies progression into `Improving`, `Stable`, or `Worsening`.
3. **Clinical Text NLP (BiLSTM):** Analyzes dermatologist notes and patient skincare diaries for treatment efficacy and adverse irritation reactions.

In [ ]:
# 1. Environment Setup & Hardware Diagnostics
import os
import sys
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, mean_absolute_error, r2_score

# Fix seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Compute Device: {device}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")

## 2. Longitudinal Dataset Formulation from PAD-UFES-20
Constructing sequential multi-visit histories from `metadata.csv` (PAD-UFES-20) and `pigmentation_report.csv` (Module 3 U-Net segmenter).

In [ ]:
from src.module4_sequence_dataset import build_longitudinal_dataset, LongitudinalSkinDataset

BASE_DIR = os.path.dirname(os.path.abspath('')) if '__file__' not in locals() else '.'
sequences, t1m, t3m, t6m, trend_labels = build_longitudinal_dataset(base_dir=BASE_DIR, seq_len=4, seed=SEED)

print(f"Total Patient Sequences: {len(sequences)}")
print(f"Sequence Tensor Shape: {sequences.shape} (T=4 past visits, D=8 clinical features)")
print(f"Target 1M Shape: {t1m.shape} (Health, Pigmentation %, Severity)")
print(f"Target 3M Shape: {t3m.shape}")
print(f"Target 6M Shape: {t6m.shape}")
print(f"Trajectory Classes: Improving={sum(trend_labels==0)}, Stable={sum(trend_labels==1)}, Worsening={sum(trend_labels==2)}")

## 3. Recurrent Deep Learning Architectures (LSTM vs GRU)
Building `SkinProgressionLSTM` with multi-horizon projection heads and `SkinProgressionGRU` for comparative evaluation.

In [ ]:
from src.module4_models import SkinProgressionLSTM, SkinProgressionGRU

# Instantiate models
lstm_model = SkinProgressionLSTM(input_dim=8, hidden_dim=64, num_layers=2, dropout=0.25).to(device)
gru_model = SkinProgressionGRU(input_dim=8, hidden_dim=64, num_layers=2, dropout=0.25).to(device)

print("SkinProgressionLSTM Architecture:")
print(lstm_model)

print("\nTotal Trainable Parameters:")
print(f"- LSTM: {sum(p.numel() for p in lstm_model.parameters() if p.requires_grad):,}")
print(f"- GRU:  {sum(p.numel() for p in gru_model.parameters() if p.requires_grad):,}")

## 4. Multi-Horizon Training & Convergence Evaluation
Training both recurrent networks with multi-task objective (MSE on 1M, 3M, 6M continuous forecasts + CrossEntropy on trajectory classification).

In [ ]:
# Train/Val/Test Splits
idx_train, idx_temp = train_test_split(np.arange(len(sequences)), test_size=0.30, random_state=SEED, stratify=trend_labels)
idx_val, idx_test = train_test_split(idx_temp, test_size=0.50, random_state=SEED, stratify=trend_labels[idx_temp])

full_ds = LongitudinalSkinDataset(sequences, t1m, t3m, t6m, trend_labels)
train_loader = DataLoader(Subset(full_ds, idx_train), batch_size=32, shuffle=True)
val_loader = DataLoader(Subset(full_ds, idx_val), batch_size=32, shuffle=False)
test_loader = DataLoader(Subset(full_ds, idx_test), batch_size=32, shuffle=False)

from train_module4 import train_progression_model, evaluate_progression_model

print("Training SkinProgressionLSTM...")
lstm_model, lstm_tr_loss, lstm_val_loss = train_progression_model(lstm_model, train_loader, val_loader, epochs=25, lr=0.001, device=device)

print("Training SkinProgressionGRU...")
gru_model, gru_tr_loss, gru_val_loss = train_progression_model(gru_model, train_loader, val_loader, epochs=25, lr=0.001, device=device)

print("Training completed!")

## 5. Comparative Performance Benchmarks & Loss Curves

In [ ]:
lstm_eval = evaluate_progression_model(lstm_model, test_loader, device=device)
gru_eval = evaluate_progression_model(gru_model, test_loader, device=device)

print(f"SkinProgressionLSTM -> Test MAE: {lstm_eval['overall_mae']:.4f} | R2: {lstm_eval['overall_r2']:.4f} | Trend Acc: {lstm_eval['trend_accuracy']*100:.2f}%")
print(f"SkinProgressionGRU  -> Test MAE: {gru_eval['overall_mae']:.4f} | R2: {gru_eval['overall_r2']:.4f} | Trend Acc: {gru_eval['trend_accuracy']*100:.2f}%")

plt.figure(figsize=(10, 4.5))
plt.plot(lstm_tr_loss, label='LSTM Train Loss', color='#2563eb', linewidth=2)
plt.plot(lstm_val_loss, label='LSTM Val Loss', color='#3b82f6', linestyle='--', linewidth=2)
plt.plot(gru_tr_loss, label='GRU Train Loss', color='#10b981', linewidth=2)
plt.plot(gru_val_loss, label='GRU Val Loss', color='#34d399', linestyle='--', linewidth=2)
plt.title('DermaVision Module 4: Multi-Horizon Forecasting Loss (LSTM vs GRU)', fontsize=12, fontweight='bold')
plt.xlabel('Epochs')
plt.ylabel('Multi-Task Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 6. Clinical Notes & Skincare Journal Analysis (BiLSTM)
Bidirectional recurrent text classifier to parse dermatologist consultation notes and patient journals.

In [ ]:
from src.module4_sequence_dataset import generate_augmented_clinical_corpus, build_vocab
from train_module4 import train_and_eval_bilstm

aug_corpus = generate_augmented_clinical_corpus(multiplier=25, seed=SEED)
vocab = build_vocab(aug_corpus)
print(f"Clinical Vocabulary Size: {len(vocab)} unique medical tokens")

bilstm_model, bilstm_acc, bilstm_trues, bilstm_preds = train_and_eval_bilstm(
    vocab, aug_corpus, device=device, epochs=18, lr=0.002
)
print(f"BiLSTM Classification Accuracy: {bilstm_acc*100:.2f}%")

cm = confusion_matrix(bilstm_trues, bilstm_preds)
plt.figure(figsize=(5.5, 4.5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Positive', 'Neutral', 'Adverse'], yticklabels=['Positive', 'Neutral', 'Adverse'])
plt.title('BiLSTM Clinical Notes Confusion Matrix')
plt.xlabel('Predicted Class')
plt.ylabel('Ground Truth Class')
plt.show()

## 7. Interactive Multi-Horizon Projections & Trend Alerts
Simulating a patient progression timeline with multi-step future projections.

In [ ]:
from src.module4_forecast import predict_skin_progression, analyze_clinical_notes

# Example patient visit sequence over 4 visits
patient_visits = [
    [0.72, 0.32, 0.28, 0.40, 0.40, 0.30, 0.33, 0.85],
    [0.75, 0.29, 0.25, 0.38, 0.38, 0.29, 0.17, 0.90],
    [0.79, 0.26, 0.21, 0.35, 0.36, 0.27, 0.17, 0.92],
    [0.82, 0.23, 0.18, 0.32, 0.35, 0.26, 0.00, 0.95]
]

forecast_result = predict_skin_progression(lstm_model, patient_visits, device=device)

print("=" * 65)
print("DERMAVISION AI — MODULE 4 PATIENT PROGRESSION REPORT")
print("=" * 65)
print(f"Projected Trajectory: {forecast_result['trajectory_category']} (Conf: {forecast_result['trajectory_confidence']}%)")
print(f"- 1 Month Horizon:  Health={forecast_result['forecast_1_month']['skin_health_score']} | Pigmentation={forecast_result['forecast_1_month']['pigmentation_percentage']}%")
print(f"- 3 Months Horizon: Health={forecast_result['forecast_3_months']['skin_health_score']} | Pigmentation={forecast_result['forecast_3_months']['pigmentation_percentage']}%")
print(f"- 6 Months Horizon: Health={forecast_result['forecast_6_months']['skin_health_score']} | Pigmentation={forecast_result['forecast_6_months']['pigmentation_percentage']}%")
print(f"Recommendation: {forecast_result['clinical_recommendation']}")

# Test BiLSTM Note
sample_note = "Patient reports dark spots on cheeks have visibly faded after 6 weeks of retinoid and sunscreen."
note_analysis = analyze_clinical_notes(bilstm_model, vocab, sample_note, device=device)
print(f"\nClinical Note Assessment: {note_analysis['feedback_category']} (Conf: {note_analysis['confidence']}%)")